In [2]:
# ============================================================
# Cell 1 — Setup + Paths (CMIP6 ↔ ERA5 correction workflow)
# ============================================================

from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict
from datetime import datetime, UTC
from pathlib import Path
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
from xgboost import XGBRegressor

from rozvidrought_datasets.grid import RozviGrid

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought")

# Historical CMIP6
CMIP6_HIST_ROOT = PROJECT_ROOT / "data" / "historical" / "cmip6"

# ERA5 / corrected reference baseline
ERA5_SIM_DIRS = {
    "pet_ref": PROJECT_ROOT / "data" / "simulated" / "era5_land" / "pet",
    "t2m_ref": PROJECT_ROOT / "data" / "simulated" / "era5_land" / "t2m",
    "d2m_ref": PROJECT_ROOT / "data" / "simulated" / "era5_land" / "d2m",
}
SM_CORRECTED_DIR = PROJECT_ROOT / "rasters" / "corrected" / "sm"

# Future CMIP6 scenarios
SCENARIO_ROOT = PROJECT_ROOT / "data" / "scenarios"
SCENARIOS = ["ssp245", "ssp370", "ssp585"]

# Outputs
ARTIFACTS_DIR = PROJECT_ROOT / "data" / "artifacts"
DATASETS_DIR = ARTIFACTS_DIR / "datasets"
MODELS_DIR = ARTIFACTS_DIR / "models"
MANIFESTS_DIR = ARTIFACTS_DIR / "manifests"
CMIP6_CORRECTED_DIR = PROJECT_ROOT / "rasters" / "corrected" / "cmip6"

for p in [DATASETS_DIR, MODELS_DIR, MANIFESTS_DIR, CMIP6_CORRECTED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CMIP6_VARIABLES = {
    "tas": "t2m_ref",
    "huss": "d2m_ref",
    "mrsos": "sm_ref",
    "rsds": "pet_ref",   # proxy path into PET-related correction context
}

TARGET_COLUMNS = ["t2m_ref", "d2m_ref", "pet_ref", "sm_ref"]

def utc_stamp() -> str:
    return datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")

print("Notebook root :", PROJECT_ROOT)
print("Historical CMIP6:", CMIP6_HIST_ROOT)
print("Reference dirs :", ERA5_SIM_DIRS)
print("Corrected SM   :", SM_CORRECTED_DIR)
print("Scenarios      :", SCENARIOS)

Notebook root : C:\Projects\Infer RozviDrought
Historical CMIP6: C:\Projects\Infer RozviDrought\data\historical\cmip6
Reference dirs : {'pet_ref': WindowsPath('C:/Projects/Infer RozviDrought/data/simulated/era5_land/pet'), 't2m_ref': WindowsPath('C:/Projects/Infer RozviDrought/data/simulated/era5_land/t2m'), 'd2m_ref': WindowsPath('C:/Projects/Infer RozviDrought/data/simulated/era5_land/d2m')}
Corrected SM   : C:\Projects\Infer RozviDrought\rasters\corrected\sm
Scenarios      : ['ssp245', 'ssp370', 'ssp585']


In [3]:
# ============================================================
# Cell 2 — Load historical CMIP6 + reference ERA5 rasters
# ============================================================

FILE_RE = re.compile(r"^(?P<var>[A-Za-z0-9_]+)_(?P<yyyymm>\d{6})\.tif$", re.IGNORECASE)

def discover_tifs(folder: Path) -> Dict[str, Path]:
    out = {}
    if not folder.exists():
        return out
    for p in folder.glob("*.tif"):
        m = FILE_RE.match(p.name)
        if m:
            out[m.group("yyyymm")] = p
    return out

def read_resample_tif(src_path: Path, dst_shape, dst_transform, dst_crs) -> np.ndarray:
    dst = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)
        reproject(
            source=arr,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=np.nan,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return dst

grid = RozviGrid()
west, south, east, north = (
    float(grid.pixel_bounds(0)[0]),
    float(grid.pixel_bounds(grid.width * grid.height - 1)[1]),
    float(grid.pixel_bounds(grid.width * grid.height - 1)[2]),
    float(grid.pixel_bounds(0)[3]),
)
dst_transform = from_bounds(west, south, east, north, grid.width, grid.height)
dst_crs = "EPSG:4326"
dst_shape = (grid.height, grid.width)

era5_index = {
    "pet_ref": discover_tifs(ERA5_SIM_DIRS["pet_ref"]),
    "t2m_ref": discover_tifs(ERA5_SIM_DIRS["t2m_ref"]),
    "d2m_ref": discover_tifs(ERA5_SIM_DIRS["d2m_ref"]),
    "sm_ref": discover_tifs(SM_CORRECTED_DIR),
}

cmip6_hist_files = {
    "tas": list((CMIP6_HIST_ROOT / "tas").glob("*.nc"))[0],
    "huss": list((CMIP6_HIST_ROOT / "huss").glob("*.nc"))[0],
    "mrsos": list((CMIP6_HIST_ROOT / "mrsos").glob("*.nc"))[0],
    "rsds": list((CMIP6_HIST_ROOT / "rsds").glob("*.nc"))[0],
}

print("ERA5 indexed months:", {k: len(v) for k, v in era5_index.items()})
print("CMIP6 historical files:", {k: v.name for k, v in cmip6_hist_files.items()})

ERA5 indexed months: {'pet_ref': 523, 't2m_ref': 523, 'd2m_ref': 523, 'sm_ref': 523}
CMIP6 historical files: {'tas': 'tas_Amon_MPI-ESM1-2-LR_historical_r1i1p1f1_gn_19810116-20141216.nc', 'huss': 'huss_Amon_MPI-ESM1-2-LR_historical_r1i1p1f1_gn_19810116-20141216.nc', 'mrsos': 'mrsos_Lmon_MPI-ESM1-2-LR_historical_r1i1p1f1_gn_19810116-20141216.nc', 'rsds': 'rsds_Amon_MPI-ESM1-2-LR_historical_r1i1p1f1_gn_19810116-20141216.nc'}


In [4]:
# ============================================================
# Cell 3 — Build historical CMIP6 ↔ ERA5 training dataset
# ============================================================

def yyyymm_from_time(t) -> str:
    ts = pd.Timestamp(t)
    return f"{ts.year}{ts.month:02d}"

def open_cmip6_var(path: Path, var_name: str) -> xr.DataArray:
    ds = xr.open_dataset(path, engine="netcdf4")
    return ds[var_name]

cmip6_hist = {
    "tas": open_cmip6_var(cmip6_hist_files["tas"], "tas"),
    "huss": open_cmip6_var(cmip6_hist_files["huss"], "huss"),
    "mrsos": open_cmip6_var(cmip6_hist_files["mrsos"], "mrsos"),
    "rsds": open_cmip6_var(cmip6_hist_files["rsds"], "rsds"),
}

cmip6_months = [yyyymm_from_time(t) for t in cmip6_hist["tas"].time.values]
common_months = [
    m for m in cmip6_months
    if m in era5_index["pet_ref"]
    and m in era5_index["t2m_ref"]
    and m in era5_index["d2m_ref"]
    and m in era5_index["sm_ref"]
]

records = []

for i, yyyymm in enumerate(common_months):
    cmip6_arrays = {}
    for var in ["tas", "huss", "mrsos", "rsds"]:
        da = cmip6_hist[var].isel(time=i)
        arr = da.values.astype(np.float32)

        src_transform = rasterio.transform.from_bounds(
            float(da.lon.values.min()),
            float(da.lat.values.min()),
            float(da.lon.values.max()),
            float(da.lat.values.max()),
            da.sizes["lon"],
            da.sizes["lat"],
        )

        dst = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=arr,
            destination=dst,
            src_transform=src_transform,
            src_crs="EPSG:4326",
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
        cmip6_arrays[var] = dst.reshape(-1)

    era5_arrays = {
        "pet_ref": read_resample_tif(era5_index["pet_ref"][yyyymm], dst_shape, dst_transform, dst_crs).reshape(-1),
        "t2m_ref": read_resample_tif(era5_index["t2m_ref"][yyyymm], dst_shape, dst_transform, dst_crs).reshape(-1),
        "d2m_ref": read_resample_tif(era5_index["d2m_ref"][yyyymm], dst_shape, dst_transform, dst_crs).reshape(-1),
        "sm_ref": read_resample_tif(era5_index["sm_ref"][yyyymm], dst_shape, dst_transform, dst_crs).reshape(-1),
    }

    df_month = pd.DataFrame({
        "yyyymm": yyyymm,
        "pixel_id": np.arange(dst_shape[0] * dst_shape[1]),
        "tas": cmip6_arrays["tas"],
        "huss": cmip6_arrays["huss"],
        "mrsos": cmip6_arrays["mrsos"],
        "rsds": cmip6_arrays["rsds"],
        "t2m_ref": era5_arrays["t2m_ref"],
        "d2m_ref": era5_arrays["d2m_ref"],
        "sm_ref": era5_arrays["sm_ref"],
        "pet_ref": era5_arrays["pet_ref"],
    })

    records.append(df_month)

hist_training_df = pd.concat(records, ignore_index=True)

out_path = DATASETS_DIR / f"cmip6_era5_training_dataset_{utc_stamp()}.parquet"
hist_training_df.to_parquet(out_path, index=False)

print("Common months:", len(common_months))
print("Rows:", len(hist_training_df))
print("Saved:", out_path)
print(hist_training_df.head())

Common months: 400
Rows: 10710000
Saved: C:\Projects\Infer RozviDrought\data\artifacts\datasets\cmip6_era5_training_dataset_20260321T095919Z.parquet
   yyyymm  pixel_id         tas      huss     mrsos        rsds     t2m_ref  \
0  198109         0  291.774353  0.003613  7.768919  234.537979  295.566101   
1  198109         1  291.781433  0.003618  7.763739  234.449997  295.494171   
2  198109         2  291.788544  0.003622  7.758557  234.362015  295.456207   
3  198109         3  291.795624  0.003626  7.753376  234.274017  295.417206   
4  198109         4  291.802734  0.003631  7.748195  234.186035  295.369476   

      d2m_ref    sm_ref   pet_ref  
0  278.061859  0.208754 -0.000590  
1  278.196564  0.205828 -0.000570  
2  278.303955  0.214803 -0.000524  
3  278.403198  0.243379 -0.000482  
4  278.435089  0.257517 -0.000475  


In [5]:
# ============================================================
# Cell 4 — Quick sample comparison: CMIP6 vs ERA5 reference
# ============================================================

sample_cols = [
    "tas", "t2m_ref",
    "huss", "d2m_ref",
    "mrsos", "sm_ref",
    "rsds", "pet_ref",
]

sample_df = hist_training_df[sample_cols].sample(
    n=5000,
    random_state=42
).copy()

summary = {}

pairs = [
    ("tas", "t2m_ref"),
    ("huss", "d2m_ref"),
    ("mrsos", "sm_ref"),
    ("rsds", "pet_ref"),
]

for x, y in pairs:
    diff = sample_df[y] - sample_df[x]
    summary[f"{x}_vs_{y}"] = {
        "cmip6_mean": float(sample_df[x].mean()),
        "ref_mean": float(sample_df[y].mean()),
        "mean_diff": float(diff.mean()),
        "abs_mean_diff": float(diff.abs().mean()),
        "corr": float(sample_df[[x, y]].corr().iloc[0, 1]),
    }

for k, v in summary.items():
    print("\n", k)
    for kk, vv in v.items():
        print(f"  {kk}: {vv:.6f}")


 tas_vs_t2m_ref
  cmip6_mean: 298.617401
  ref_mean: 294.794861
  mean_diff: -3.822520
  abs_mean_diff: 7.668609
  corr: 0.012664

 huss_vs_d2m_ref
  cmip6_mean: 0.004363
  ref_mean: 285.042542
  mean_diff: 285.038208
  abs_mean_diff: 285.038208
  corr: -0.090194

 mrsos_vs_sm_ref
  cmip6_mean: 5.388061
  ref_mean: 0.221062
  mean_diff: -5.166998
  abs_mean_diff: 5.166998
  corr: -0.020168

 rsds_vs_pet_ref
  cmip6_mean: 287.733459
  ref_mean: -0.000362
  mean_diff: -287.733795
  abs_mean_diff: 287.733795
  corr: 0.073190


In [6]:
# ============================================================
# Cell 5 — Fix variable alignment (create correct targets)
# ============================================================

df = hist_training_df.copy()

# 1) Soil moisture: convert kg/m² → volumetric fraction (~0–1)
# Assume 0–10 cm layer, density of water ≈ 1000 kg/m³
# thickness = 0.1 m

df["mrsos_vol"] = df["mrsos"] / (1000 * 0.1)

# 2) Temperature already comparable (Kelvin)
df["tas_target"] = df["t2m_ref"]

# 3) Humidity: keep dewpoint as proxy target
df["huss_target"] = df["d2m_ref"]

# 4) Radiation: use rsds directly as its own reference
df["rsds_target"] = df["rsds"]

print("Targets created:")
print([
    "tas_target",
    "huss_target",
    "mrsos_vol",
    "rsds_target",
])

print("\nSample:")
print(df[
    [
        "tas",
        "tas_target",
        "mrsos",
        "mrsos_vol",
        "rsds",
    ]
].head())

Targets created:
['tas_target', 'huss_target', 'mrsos_vol', 'rsds_target']

Sample:
          tas  tas_target     mrsos  mrsos_vol        rsds
0  291.774353  295.566101  7.768919   0.077689  234.537979
1  291.781433  295.494171  7.763739   0.077637  234.449997
2  291.788544  295.456207  7.758557   0.077586  234.362015
3  291.795624  295.417206  7.753376   0.077534  234.274017
4  291.802734  295.369476  7.748195   0.077482  234.186035


In [7]:
# ============================================================
# Cell 6 — Keep only variables that are truly comparable now
# ============================================================

train_df = df[[
    "yyyymm", "pixel_id",
    "tas", "tas_target",
    "mrsos_vol", "sm_ref"
]].dropna().copy()

print("Ready for training:")
print(train_df.columns.tolist())
print("Rows:", len(train_df))
print(train_df.head())

Ready for training:
['yyyymm', 'pixel_id', 'tas', 'tas_target', 'mrsos_vol', 'sm_ref']
Rows: 10710000
   yyyymm  pixel_id         tas  tas_target  mrsos_vol    sm_ref
0  198109         0  291.774353  295.566101   0.077689  0.208754
1  198109         1  291.781433  295.494171   0.077637  0.205828
2  198109         2  291.788544  295.456207   0.077586  0.214803
3  198109         3  291.795624  295.417206   0.077534  0.243379
4  198109         4  291.802734  295.369476   0.077482  0.257517


In [8]:
# ============================================================
# Cell 7 — Train CMIP6 → ERA5 correction models
# ============================================================

from xgboost import XGBRegressor
from pathlib import Path
import joblib
from datetime import datetime, UTC

ARTIFACTS_DIR = (
    Path("data")
    / "artifacts"
    / "models"
)

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")

# -------------------------
# Temperature model
# -------------------------

print("\nTraining temperature correction model...")

tas_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

tas_model.fit(
    train_df[["tas"]],
    train_df["tas_target"]
)

tas_model_path = (
    ARTIFACTS_DIR
    / f"cmip6_bias_model_tas_{timestamp}.joblib"
)

joblib.dump(tas_model, tas_model_path)

print("Saved:", tas_model_path)

# -------------------------
# Soil moisture model
# -------------------------

print("\nTraining soil moisture correction model...")

sm_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

sm_model.fit(
    train_df[["mrsos_vol"]],
    train_df["sm_ref"]
)

sm_model_path = (
    ARTIFACTS_DIR
    / f"cmip6_bias_model_mrsos_{timestamp}.joblib"
)

joblib.dump(sm_model, sm_model_path)

print("Saved:", sm_model_path)

print("\n===================================")
print("Models trained successfully.")


Training temperature correction model...
Saved: data\artifacts\models\cmip6_bias_model_tas_20260321T102243Z.joblib

Training soil moisture correction model...
Saved: data\artifacts\models\cmip6_bias_model_mrsos_20260321T102243Z.joblib

Models trained successfully.


In [ ]:
# ============================================================
# Validation section
# Goal:
# 1) read corrected CMIP6 tas/sm rasters
# 2) check month count, CRS, shape, value ranges
# 3) compare against raw CMIP6-derived monthly outputs if needed
# This is post-correction validation only.Do this after training and running inference on
# the correction model
# ============================================================

In [8]:
# ============================================================
# Cell — Self-contained validation of corrected CMIP6 outputs
# ============================================================

from pathlib import Path
import re
import numpy as np
import rasterio

# ---- Configuration (self-contained) ----

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought")

SCENARIOS = [
    "ssp245",
    "ssp370",
    "ssp585"
]

CMIP6_CORRECTED_ROOT = (
    PROJECT_ROOT
    / "rasters"
    / "corrected"
    / "cmip6"
)

FILE_RE = re.compile(
    r"^(?P<var>[A-Za-z0-9_]+)_corrected_(?P<yyyymm>\d{6})\.tif$",
    re.IGNORECASE
)

# ---- Helper ----

def discover_corrected(folder: Path):
    out = {}
    if not folder.exists():
        return out

    for p in folder.glob("*.tif"):
        m = FILE_RE.match(p.name)
        if m:
            out[m.group("yyyymm")] = p

    return out

# ---- Validation ----

summary = []

for scenario in SCENARIOS:

    for var in ["tas", "sm"]:

        var_dir = CMIP6_CORRECTED_ROOT / scenario / var

        files = discover_corrected(var_dir)

        if not files:

            summary.append({
                "scenario": scenario,
                "variable": var,
                "status": "missing"
            })

            continue

        sample_month = sorted(files.keys())[0]
        sample_file = files[sample_month]

        with rasterio.open(sample_file) as src:

            arr = src.read(1).astype(np.float32)

            nodata = src.nodata

            if nodata is not None:
                arr[arr == nodata] = np.nan

            summary.append({
                "scenario": scenario,
                "variable": var,
                "status": "ok",
                "months_found": len(files),
                "sample_month": sample_month,
                "sample_file": sample_file.name,
                "crs": src.crs.to_string() if src.crs else None,
                "shape": (src.height, src.width),
                "min": float(np.nanmin(arr)),
                "max": float(np.nanmax(arr)),
                "mean": float(np.nanmean(arr)),
            })

print("===================================")
print("CMIP6 CORRECTED OUTPUT VALIDATION")
print("===================================")

for item in summary:
    print(item)

CMIP6 CORRECTED OUTPUT VALIDATION
{'scenario': 'ssp245', 'variable': 'tas', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'tas_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (153, 175), 'min': 293.9447021484375, 'max': 295.28173828125, 'mean': 294.7542419433594}
{'scenario': 'ssp245', 'variable': 'sm', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'sm_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (153, 175), 'min': 0.19869381189346313, 'max': 0.24042285978794098, 'mean': 0.22365528345108032}
{'scenario': 'ssp370', 'variable': 'tas', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'tas_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (153, 175), 'min': 293.9447021484375, 'max': 295.28173828125, 'mean': 294.8109436035156}
{'scenario': 'ssp370', 'variable': 'sm', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'sm_corrected_202601.tif', 'crs': 'EPSG:432

In [9]:
# ============================================================
# Cell — Save validation manifest for corrected CMIP6 outputs
# ============================================================

from pathlib import Path
from datetime import datetime, UTC
import json
import re

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought")

SCENARIOS = ["ssp245", "ssp370", "ssp585"]

CMIP6_CORRECTED_ROOT = (
    PROJECT_ROOT
    / "rasters"
    / "corrected"
    / "cmip6"
)

MANIFEST_DIR = (
    PROJECT_ROOT
    / "data"
    / "artifacts"
    / "manifests"
)

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

FILE_RE = re.compile(
    r"^(?P<var>[A-Za-z0-9_]+)_corrected_(?P<yyyymm>\d{6})\.tif$",
    re.IGNORECASE
)

timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")

manifest = {
    "run_id": f"cmip6_tempsoil_validation_{timestamp}",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "dataset": "CMIP6 corrected temperature and soil moisture",
    "time_span": "2026-2050",
    "scenarios": {},
    "notes": [
        "Validated corrected CMIP6 tas and soil moisture outputs",
        "All scenarios contain 300 monthly rasters",
        "Outputs aligned to Rozvi grid (153 x 175)",
        "CRS: EPSG:4326"
    ]
}

for scenario in SCENARIOS:

    scenario_record = {}

    for var in ["tas", "sm"]:

        var_dir = CMIP6_CORRECTED_ROOT / scenario / var

        months = []

        if var_dir.exists():

            for f in var_dir.glob("*.tif"):

                m = FILE_RE.match(f.name)

                if m:
                    months.append(m.group("yyyymm"))

        months = sorted(months)

        scenario_record[var] = {
            "files_found": len(months),
            "first_month": months[0] if months else None,
            "last_month": months[-1] if months else None,
            "expected_months": 300,
            "status": "ok" if len(months) == 300 else "incomplete"
        }

    manifest["scenarios"][scenario] = scenario_record

manifest_path = (
    MANIFEST_DIR
    / f"cmip6_tempsoil_validation_manifest_2026_2050_{timestamp}.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:

    json.dump(manifest, f, indent=2)

print("===================================")
print("Validation manifest saved:")
print(manifest_path)
print("===================================")

Validation manifest saved:
C:\Projects\Infer RozviDrought\data\artifacts\manifests\cmip6_tempsoil_validation_manifest_2026_2050_20260321T111439Z.json
